## Telco Customer Churn — Preprocessing & Training Data Development

**Author:** Allison Schiltz

> Dataset reference: *Telco Customer Churn* (Kaggle).
https://www.kaggle.com/datasets/blastchar/telco-customer-churn/code

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


import matplotlib.pyplot as plt
import seaborn as sns

import featuretools as ft

# Reproducibility
SEED = 42
np.random.seed(SEED)

## Outline

This notebook will cover the following steps for preparing data for modeling:


1. **Load Cleaned Data**

   - Import pandas and load the cleaned dataset from `data/processed/telco_wrangling_cleaned.csv` using an absolute path.


2. **Identify Feature Types**

   - Use `references/telco_column_role_manifest_template.csv` to distinguish categorical and numeric columns.


3. **Create Dummy/Indicator Features**

   - Use `pandas.get_dummies()` for categorical variables.
   - Drop the original categorical columns.
   - Document columns excluded from encoding (e.g., ID, target).


4. **Split Data into Train/Test Sets**

   - Use `sklearn.model_selection.train_test_split` after encoding and dropping ID columns.
   - Stratify by the target variable (`Churn`) to preserve class balance.
   - Set `SEED = 42` for reproducibility.


5. **Standardize Numeric Features**

   - Use `sklearn`'s `StandardScaler` to scale numeric columns.
   - Fit the scaler on the training data, then transform both train and test sets.


6. **Save/Document Outputs**

   - Save the processed train/test sets to `data/processed/`.

## Import Libraries

### 1. Load Cleaned Data
   - Import pandas and load the cleaned dataset from `data/processed/telco_wrangling_cleaned.csv` using an absolute path.

In [2]:
telco_clean = pd.read_csv('../data/processed/telco_wrangling_cleaned.csv')
print(telco_clean.head())

   customerID  gender SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female            No     Yes         No       1           No   
1  5575-GNVDE    Male            No      No         No      34          Yes   
2  3668-QPYBK    Male            No      No         No       2          Yes   
3  7795-CFOCW    Male            No      No         No      45           No   
4  9237-HQITU  Female            No      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ...  \
0  No phone service             DSL             No  ...   
1                No             DSL            Yes  ...   
2                No             DSL            Yes  ...   
3  No phone service             DSL            Yes  ...   
4                No     Fiber optic             No  ...   

               PaymentMethod MonthlyCharges TotalCharges Churn  \
0           Electronic check          29.85        29.85    No   
1               Mailed check          

### 2. Identify Feature Types
   - Use `references/telco_column_role_manifest_template.csv` to distinguish categorical and numeric columns.

In [3]:
column_roles = pd.read_csv('../references/telco_column_role_manifest.csv')
print(column_roles)

                        column_name         role  \
0                        customerID           id   
1                             Churn       target   
2                            tenure      numeric   
3                    MonthlyCharges      numeric   
4                      TotalCharges      numeric   
5                            gender  categorical   
6                     SeniorCitizen  categorical   
7                           Partner  categorical   
8                        Dependents  categorical   
9                      PhoneService  categorical   
10                    MultipleLines  categorical   
11                  InternetService  categorical   
12                   OnlineSecurity  categorical   
13                     OnlineBackup  categorical   
14                 DeviceProtection  categorical   
15                      TechSupport  categorical   
16                      StreamingTV  categorical   
17                  StreamingMovies  categorical   
18          

### 3. Create Dummy/Indicator Features
   - Use `pandas.get_dummies()` 
   - Drop the original categorical columns.
   - Drop ID customerID column
   - Document columns excluded from encoding (e.g., ID, target).

In [4]:
# Create list of categorical variables
categorical_cols = column_roles[column_roles['role'] == 'categorical']['column_name'].tolist()
print(categorical_cols)

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group_eng', 'monthly_charges_bin_eng', 'is_month_to_month_eng', 'short_tenure_month_to_month_eng', 'has_streaming_eng']


In [5]:
# One-hot encode categorical variables using pandas.get_dummies(). The columns specified for encoding are automatically dropped.

telco_encoded = pd.get_dummies(telco_clean, columns=categorical_cols, drop_first=True)

print(telco_encoded.head())

   customerID  tenure  MonthlyCharges  TotalCharges Churn  total_services_eng  \
0  7590-VHVEG       1           29.85         29.85    No                   1   
1  5575-GNVDE      34           56.95       1889.50    No                   3   
2  3668-QPYBK       2           53.85        108.15   Yes                   3   
3  7795-CFOCW      45           42.30       1840.75    No                   3   
4  9237-HQITU       2           70.70        151.65   Yes                   1   

   gender_Male  SeniorCitizen_Yes  Partner_Yes  Dependents_Yes  ...  \
0        False              False         True           False  ...   
1         True              False        False           False  ...   
2         True              False        False           False  ...   
3         True              False        False           False  ...   
4        False              False        False           False  ...   

   PaymentMethod_Mailed check  tenure_group_eng_13-24  tenure_group_eng_25-48  \
0    

In [6]:
# Drop customerID column as it is not needed for modeling
telco_encoded = telco_encoded.drop('customerID'
                                   , axis = 1)
print(telco_encoded.head())


   tenure  MonthlyCharges  TotalCharges Churn  total_services_eng  \
0       1           29.85         29.85    No                   1   
1      34           56.95       1889.50    No                   3   
2       2           53.85        108.15   Yes                   3   
3      45           42.30       1840.75    No                   3   
4       2           70.70        151.65   Yes                   1   

   gender_Male  SeniorCitizen_Yes  Partner_Yes  Dependents_Yes  \
0        False              False         True           False   
1         True              False        False           False   
2         True              False        False           False   
3         True              False        False           False   
4        False              False        False           False   

   PhoneService_Yes  ...  PaymentMethod_Mailed check  tenure_group_eng_13-24  \
0             False  ...                       False                   False   
1              True  ...    

In [7]:
# Save the encoded dataset
telco_encoded.to_csv('../data/processed/telco_churn_training_data.csv', index=False)

# 4. Split Data into Train/Test Sets

   - Use `sklearn.model_selection.train_test_split`.
   - Stratify by the target variable (`Churn`) to preserve class balance.
   - Set `SEED = 42` for reproducibility.

In [8]:
# Check alignment and columns before splitting
print("telco_encoded shape:", telco_encoded.shape)
print("telco_clean shape:", telco_clean.shape)
print("'Churn' in telco_encoded columns:", 'Churn' in telco_encoded.columns)
print("Indexes equal:", (telco_encoded.index == telco_clean.index).all())

telco_encoded shape: (7032, 41)
telco_clean shape: (7032, 27)
'Churn' in telco_encoded columns: True
Indexes equal: True


In [9]:
# Split the data into training and testing sets
X = telco_encoded
y = telco_clean['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, random_state=42, 
                                                    stratify=y)

In [10]:
# Drop target variable from feature sets
X_train = X_train.drop('Churn', axis=1)
X_test = X_test.drop('Churn', axis=1)

In [11]:
print(X_train.head())
print(type(X_train))

      tenure  MonthlyCharges  TotalCharges  total_services_eng  gender_Male  \
1408      65           94.55       6078.75                   6         True   
6992      26           35.75       1022.50                   2         True   
3349      68           90.20       6297.65                   5        False   
4486       3           84.30        235.05                   3         True   
3535      49           40.65       2070.75                   2        False   

      SeniorCitizen_Yes  Partner_Yes  Dependents_Yes  PhoneService_Yes  \
1408              False         True            True              True   
6992              False        False           False             False   
3349              False         True           False              True   
4486              False        False           False              True   
3535              False         True           False             False   

      MultipleLines_No phone service  ...  PaymentMethod_Mailed check  \
1408   

In [12]:
# Convert target to numeric labels
y_train = y_train.map({'No': 0, 'Yes': 1})
y_test = y_test.map({'No': 0, 'Yes': 1})

### 5. Standardize Numeric Features
   - Use `sklearn`'s `StandardScaler` to scale numeric columns.
   - Fit the scaler only on the training data (after splitting).

In [13]:
# Select numeric columns for scaling
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
X_train_num = X_train[numeric_cols]
X_test_num = X_test[numeric_cols]


In [14]:
# Instantiate the scaler
scaler = StandardScaler()

# Fit the scaler on the training data
X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)

In [15]:
print(X_train_num[:5])
print(X_test_num[:5])

[[ 1.3218163   0.98155578  1.6599004   1.26994787]
 [-0.26741023 -0.97154551 -0.56225219 -0.67186844]
 [ 1.4440645   0.83706615  1.75610395  0.78449379]
 [-1.20464639  0.6410917  -0.90832567 -0.18641437]
 [ 0.66982593 -0.80878707 -0.10156068 -0.67186844]]
[[ 1.07731992  0.36373803  0.98467365  0.78449379]
 [-1.0416488   0.45009965 -0.78179757 -0.18641437]
 [ 0.87357292 -1.49137604 -0.53722344 -1.15732252]
 [-1.24539579 -1.47310724 -0.99461881 -1.15732252]
 [ 1.56631269  1.33364547  2.30869204  1.26994787]]


In [16]:
# Convert the scaled arrays back to DataFrames
X_train_num = pd.DataFrame(X_train_num, 
                           columns=numeric_cols, 
                           index=X_train.index)
X_test_num = pd.DataFrame(X_test_num, 
                          columns=numeric_cols, 
                          index=X_test.index)

### 6. Combine scaled numerica and encoded categorical features for train and test sets

In [17]:
# Get categorical columns (already one-hot encoded)
cat_cols = [col for col in X_train.columns if col not in numeric_cols]
X_train_cat = X_train[cat_cols]
X_test_cat = X_test[cat_cols]

# Concatenate scaled numeric and categorical columns
X_train_final = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_final = pd.concat([X_test_num, X_test_cat], axis=1)

In [18]:
print(X_train_cat.shape)
print(X_test_cat.shape)
print(X_train_final.shape)
print(X_test_final.shape)

(5625, 36)
(1407, 36)
(5625, 40)
(1407, 40)


### 7. Save final combined train/test sets and targets

   - Save the processed train/test sets to `data/processed/`.

In [19]:
X_train_num.to_csv('../data/processed/X_train_num.csv', index=True)
X_test_num.to_csv('../data/processed/X_test_num.csv', index=True)
y_train.to_csv('../data/processed/y_train.csv', index=True)
y_test.to_csv('../data/processed/y_test.csv', index=True)
X_train_final.to_csv('../data/processed/X_train_final.csv', index=True)
X_test_final.to_csv('../data/processed/X_test_final.csv', index=True)